In [75]:
import os
import sys

# Keep Spark driver and workers on the same Python minor version.
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["PYSPARK_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F 
from pyspark.sql.types import (StructType, StructField, IntegerType, StringType)

spark = (
    SparkSession.builder
    .appName("bdpa-clickflow-eclothing-recommender")
    .config("spark.sql.shuffle.partitions", "32")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.driver.memory", "6g")
    .config("spark.executor.memory", "6g")
    .config("spark.memory.fraction", "0.6")
    .config("spark.pyspark.driver.python", sys.executable)
    .config("spark.pyspark.python", sys.executable)
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

raw = (
    spark.read
        .option("sep", ";")
        .option("header", "true")
        .csv("../dataset/e_shop_clothing_2008.csv")
        .toDF(
              "year", "month", "day", "order", "country",
        "session_id", "category", "item_id", "colour",
        "location", "photo_type", "price", "price_above_avg", "page"
        )
        .withColumn("year",            F.col("year").cast(IntegerType()))
        .withColumn("month",           F.col("month").cast(IntegerType()))
        .withColumn("day",             F.col("day").cast(IntegerType()))
        .withColumn("order",           F.col("order").cast(IntegerType()))
        .withColumn("country",         F.col("country").cast(IntegerType()))
        .withColumn("session_id",      F.col("session_id").cast(IntegerType()))
        .withColumn("category",        F.col("category").cast(IntegerType()))
        .withColumn("colour",          F.col("colour").cast(IntegerType()))
        .withColumn("location",        F.col("location").cast(IntegerType()))
        .withColumn("photo_type",      F.col("photo_type").cast(IntegerType()))
        .withColumn("price",           F.col("price").cast(IntegerType()))
        .withColumn("price_above_avg", F.col("price_above_avg").cast(IntegerType()))
        .withColumn("page",            F.col("page").cast(IntegerType()))
) 
    

assert raw.count() == 165474
assert raw.filter(F.col("session_id").isNull()).count() == 0
assert raw.filter(F.col("item_id").isNull()).count() == 0
assert raw.select("month").distinct().count() == 5 # april to august
assert raw.select("category").distinct().count() == 4 # trousers, skirts, blouses, sale
assert raw.select("item_id").distinct().count() == 217


print("All validation checks passed.")
raw.printSchema()
raw.show(5)


All validation checks passed.
root
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- order: integer (nullable = true)
 |-- country: integer (nullable = true)
 |-- session_id: integer (nullable = true)
 |-- category: integer (nullable = true)
 |-- item_id: string (nullable = true)
 |-- colour: integer (nullable = true)
 |-- location: integer (nullable = true)
 |-- photo_type: integer (nullable = true)
 |-- price: integer (nullable = true)
 |-- price_above_avg: integer (nullable = true)
 |-- page: integer (nullable = true)

+----+-----+---+-----+-------+----------+--------+-------+------+--------+----------+-----+---------------+----+
|year|month|day|order|country|session_id|category|item_id|colour|location|photo_type|price|price_above_avg|page|
+----+-----+---+-----+-------+----------+--------+-------+------+--------+----------+-----+---------------+----+
|2008|    4|  1|    1|     29|         1|       1|    A13|     1|  

In [76]:
session_lengths = (
    raw.groupBy("session_id")
        .agg(F.count("*").alias("n_clicks"))
)

session_lengths.describe("n_clicks").show()

n_single = session_lengths.filter(F.col("n_clicks") == 1).count()

n_total = session_lengths.count() 
print(f"Single click session: {n_single}/{n_total} ({(n_single/n_total * 100)} %)")


# item popularity

item_freq = (
    raw.groupBy('item_id') 
        .agg(F.count("*").alias("views"))
        .orderBy(F.desc("views"))
)

item_freq.show(10)


n_sessions = raw.select("session_id").distinct().count() 
n_items = raw.select("item_id").distinct().count() 
n_pairs = raw.select("session_id", "item_id").distinct().count() 
sparsity = 1 - n_pairs / (n_sessions * n_items)

print(f"Utility matrix: {n_sessions} x {n_items}")
print(f"Observed pairs: {n_pairs}")
print(f"Sparsity: {sparsity:.4%}")


# temporal distribution 
raw.groupBy("month").count().orderBy("month").show() 

raw.groupBy("category").count().orderBy("category").show()

(
    raw.groupBy("session_id", "country")
    .count() 
    .groupBy("country")
    .agg(F.countDistinct("session_id").alias("n_sessions"))
    .orderBy(F.desc("n_sessions"))
    .show(10)
)




+-------+------------------+
|summary|          n_clicks|
+-------+------------------+
|  count|             24026|
|   mean|6.8872887704986265|
| stddev|   8.9951606320672|
|    min|                 1|
|    max|               195|
+-------+------------------+

Single click session: 5042/24026 (20.985598934487637 %)
+-------+-----+
|item_id|views|
+-------+-----+
|     B4| 3579|
|     A2| 3013|
|    A11| 2789|
|     P1| 2681|
|    B10| 2566|
|     A4| 2522|
|    A15| 2489|
|     A5| 2354|
|    A10| 2280|
|     A1| 2265|
+-------+-----+
only showing top 10 rows

Utility matrix: 24026 x 217
Observed pairs: 148345
Sparsity: 97.1547%
+-----+-----+
|month|count|
+-----+-----+
|    4|48199|
|    5|35654|
|    6|32242|
|    7|35231|
|    8|14148|
+-----+-----+

+--------+-----+
|category|count|
+--------+-----+
|       1|49742|
|       2|38408|
|       3|38577|
|       4|38747|
+--------+-----+

+-------+----------+
|country|n_sessions|
+-------+----------+
|     29|     19582|
|      9|     

In [77]:
from pyspark.sql import functions as F 
from pyspark.sql.window import Window 

w_desc = Window.partitionBy("session_id").orderBy(F.desc("order"))

raw_with_rank = raw.withColumn("click_rank", F.row_number().over(w_desc))

ground_truth_full = (
    raw_with_rank
    .filter(F.col("click_rank") == 1)
    .select("session_id", "item_id")
)

train_raw_full = raw_with_rank.filter(F.col("click_rank") > 1)

session_train_stats = (
    train_raw_full.groupBy("session_id")
        .agg(F.count("*").alias("n_train_clicks"))
)

valid_sessions = (
    session_train_stats
    .filter(F.col("n_train_clicks") >= 2)
    .select("session_id")
)

cold_sessions = (
    session_train_stats
    .filter(F.col("n_train_clicks") == 1)
    .select("session_id")
)

train_raw = train_raw_full.join(valid_sessions, on="session_id", how="inner")
ground_truth = ground_truth_full.join(valid_sessions, on="session_id", how="inner")

cold_train_raw = (
    train_raw_full
    .join(cold_sessions, on="session_id", how="inner")
    .select("session_id", "item_id", "order")
)

cold_ground_truth = (
    ground_truth_full
    .join(cold_sessions, on="session_id", how="inner")
    .select("session_id", "item_id")
)

print(f"Warm sessions in training: {train_raw.select('session_id').distinct().count()}")
print(f"Warm ground truth pairs:   {ground_truth.count()}")
print(f"Cold-start sessions (1 observed click): {cold_sessions.count()}")
print(f"Cold ground-truth pairs:               {cold_ground_truth.count()}")



Warm sessions in training: 15664
Warm ground truth pairs:   15664
Cold-start sessions (1 observed click): 3320
Cold ground-truth pairs:               3320


In [78]:
TOP_K = 10
def exclude_seen_items(recs_df, seen_items_df, score_col="score", k=TOP_K):
    filtered = (
        recs_df.alias("c")
        .join(
            seen_items_df.alias("seen"),
            (F.col("c.session_idx") == F.col("seen.session_idx"))
            & (F.col("c.item_idx") == F.col("seen.seen_item_idx")),
            how="left_anti",
        )
    )

    return (
        filtered
        .withColumn(
            "rank",
            F.row_number().over(
                Window.partitionBy("session_idx").orderBy(F.desc(score_col), F.asc("item_idx"))
            ),
        )
        .filter(F.col("rank") <= k)
    )


def evaluate_recommendations(recs_df, ground_truth_df, k=TOP_K):
    eval_sessions = ground_truth_df.select("session_idx").distinct()

    top_k = (
        recs_df.filter(F.col("rank") <= k)
        .withColumn(
            "dedupe_rank",
            F.row_number().over(
                Window.partitionBy("session_idx", "item_idx").orderBy(F.col("rank"))
            ),
        )
        .filter(F.col("dedupe_rank") == 1)
        .drop("dedupe_rank")
    )

    hits = (
        top_k.join(
            ground_truth_df.withColumn("hit", F.lit(1.0)),
            on=["session_idx", "item_idx"],
            how="left",
        )
        .fillna(0, subset=["hit"])
    )

    n_rel = (
        ground_truth_df.groupBy("session_idx")
        .agg(F.count("*").alias("n_relevant"))
    )

    first_hits = (
        hits.filter(F.col("hit") == 1.0)
        .groupBy("session_idx")
        .agg(F.min("rank").alias("first_hit_rank"))
    )

    mrr_scores = (
        eval_sessions
        .join(first_hits, on="session_idx", how="left")
        .withColumn(
            "rr",
            F.when(F.col("first_hit_rank").isNotNull(), 1.0 / F.col("first_hit_rank")).otherwise(0.0),
        )
    )

    pr = (
        eval_sessions
        .join(hits.groupBy("session_idx").agg(F.sum("hit").alias("n_hits")), on="session_idx", how="left")
        .fillna(0.0, subset=["n_hits"])
        .join(n_rel, on="session_idx", how="left")
        .fillna(0.0, subset=["n_relevant"])
        .withColumn("precision", F.col("n_hits") / F.lit(k))
        .withColumn(
            "recall",
            F.when(F.col("n_relevant") > 0, F.col("n_hits") / F.col("n_relevant")).otherwise(0.0),
        )
    )

    results = {
        f"MRR@{k}": mrr_scores.agg(F.mean("rr")).collect()[0][0],
        f"Precision@{k}": pr.agg(F.mean("precision")).collect()[0][0],
        f"Recall@{k}": pr.agg(F.mean("recall")).collect()[0][0],
    }
    return results


def diversity_metrics(recs_df, train_df, n_catalog_items=217):
    n_unique_recommended = recs_df.select("item_idx").distinct().count()
    coverage = n_unique_recommended / n_catalog_items

    total_train_clicks = train_df.agg(F.sum("click_count").alias("n")).collect()[0]["n"]
    item_self_information = (
        train_df.groupBy("item_idx")
        .agg(F.sum("click_count").alias("train_clicks"))
        .withColumn("p_item", F.col("train_clicks") / F.lit(float(total_train_clicks)))
        .withColumn("self_information", -F.log2(F.col("p_item")))
        .select("item_idx", "self_information")
    )
    novelty = (
        recs_df.filter(F.col("rank") <= TOP_K)
        .join(item_self_information, on="item_idx", how="left")
        .agg(F.mean("self_information").alias("novelty"))
        .collect()[0]["novelty"]
    )

    return {
        "Coverage": coverage,
        "UniqueRecommendedItems": n_unique_recommended,
        "Novelty@10": novelty,
    }


In [79]:
import numpy as np 
from pyspark.ml.feature import StringIndexer 
from pyspark.ml.recommendation import ALS
from pyspark.sql import functions as F 
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, IntegerType, FloatType, ArrayType


TOP_K = 10
ALS_RANK = 20
ALS_ALPHA = 80.0
ALS_REG_PARAM = 0.1
SPLIT_MONTH = 7

session_first_month = (
    raw.groupBy("session_id")
        .agg(F.min("month").alias("first_month"))
)


train_session_ids = (
    session_first_month.filter(F.col("first_month") < SPLIT_MONTH).select("session_id")
)

test_session_ids = (
    session_first_month.filter(F.col("first_month") >= SPLIT_MONTH).select("session_id")
)

train_raw_split = raw.join(train_session_ids, on="session_id", how="inner")
test_raw_split = raw.join(test_session_ids, on="session_id", how="inner")

print(f"Train sessions: {train_session_ids.count()}")
print(f"Test sessions: {test_session_ids.count()}")

# leave last one out within test sessions 

w_desc = Window.partitionBy("session_id").orderBy(F.desc("order"))
test_ranked = test_raw_split.withColumn("click_rank", F.row_number().over(w_desc))


test_ground_truth_raw = (
    test_ranked.filter(F.col("click_rank") == 1).select("session_id", "item_id")
)

test_context_raw = (
    test_ranked.filter(F.col("click_rank") > 1).select("session_id", "item_id", "order")
)


valid_test_sessions = (
    test_context_raw
        .groupBy("session_id")
        .agg(F.count("*").alias("n_context"))
        .filter(F.col("n_context") >= 1)
        .select("session_id")
)


test_ground_truth_raw = test_ground_truth_raw.join(valid_test_sessions, on="session_id", how="inner")
test_context_raw = test_context_raw.join(valid_test_sessions, on="session_id", how="inner")


print(f"Evaluable test sessions: {valid_test_sessions.count()}")



# build training interaction matrix
train_interactions = (
    train_raw_split
        .groupBy("session_id", "item_id")
        .agg(F.count("*").alias("click_count"))
)


item_indexer = StringIndexer(inputCol="item_id", outputCol="item_idx").fit(raw.select("item_id"))

train_idx = (
    item_indexer.transform(train_interactions)
        .withColumn("item_idx", F.col("item_idx").cast("int"))
        .withColumn("session_idx", F.col("session_id").cast("int"))
)

train_idx.cache() 


als = ALS(
    rank = ALS_RANK,
    maxIter = 15,
    regParam = ALS_REG_PARAM, 
    alpha = ALS_ALPHA, 
    implicitPrefs=True, 
    userCol="session_idx",
    itemCol="item_idx", 
    ratingCol="click_count",
    coldStartStrategy="drop",
    seed=42
)

als_model = als.fit(train_idx)

# extract item factor matrix
item_factors_pd = als_model.itemFactors.orderBy("id").toPandas()
num_items = int(item_factors_pd["id"].max()) + 1
Y = np.zeros((num_items, ALS_RANK), dtype=np.float32)
for _, row in item_factors_pd.iterrows():
    Y[int(row["id"])] = row["features"]

YTY = Y.T @ Y 

def fold_in_session(grp_df, Y, YTY, alpha=ALS_ALPHA, reg=ALS_REG_PARAM):
    item_idxs = grp_df["item_idx"].values
    clicks = grp_df["click_count"].values.astype(np.float32)
    confidence = 1.0 + alpha * clicks 
    Y_s = Y[item_idxs] 
    A = YTY.copy() 
    A += Y_s.T @ (Y_s * (confidence - 1.0)[:, None])
    A += reg * np.eye(Y.shape[1], dtype=np.float32)
    b = Y_s.T @ confidence
    return np.linalg.solve(A, b).astype(np.float32)

test_context_interactions = (
    test_context_raw
    .groupBy("session_id", "item_id")
    .agg(F.count("*").alias("click_count"))
)

test_context_pd = (
    item_indexer.transform(test_context_interactions)
        .withColumn("item_idx", F.col("item_idx").cast("int"))
        .select("session_id", "item_idx", "click_count")
        .toPandas()
)

session_score_rows = []
for session_id, grp in test_context_pd.groupby("session_id"):
    u_vec = fold_in_session(grp, Y, YTY)
    scores = (Y @ u_vec).tolist() 
    session_score_rows.append((int(session_id), scores))

scores_schema = StructType([
    StructField("session_id", IntegerType(), False),
    StructField("scores", ArrayType(FloatType()), False)
])
scores_spark = spark.createDataFrame(session_score_rows, schema=scores_schema)

als_recs_test = (
    scores_spark
        .select(
            F.col("session_id").cast("int").alias("session_idx"),
            F.posexplode("scores").alias("item_idx", "score"),
        )
        .withColumn(
            "rank",
            F.row_number().over(
                Window.partitionBy("session_idx").orderBy(F.desc("score"), F.asc("item_idx"))
            ),
        )
)

test_ground_truth_idx = (
    item_indexer.transform(test_ground_truth_raw)
    .withColumn("item_idx", F.col("item_idx").cast("int"))
    .withColumn("session_idx", F.col("session_id").cast("int"))
    .select("session_idx", "item_idx")
)

test_seen_idx = (
    item_indexer.transform(test_context_raw.select("session_id", "item_id"))
    .withColumn("item_idx", F.col("item_idx").cast("int"))
    .withColumn("session_idx", F.col("session_id").cast("int"))
    .withColumnRenamed("item_idx", "seen_item_idx")
    .select("session_idx", "seen_item_idx")
)

als_recs_filtered = exclude_seen_items(
    als_recs_test, test_seen_idx, score_col="score", k=TOP_K
)

# evaluate
als_split_metrics = evaluate_recommendations(
    als_recs_filtered, test_ground_truth_idx, k=TOP_K
)
als_split_div = diversity_metrics(als_recs_filtered, train_idx)

# baseline evaluation on same split
item_popularity_split = train_idx.groupBy("item_idx").agg(F.sum("click_count").alias("score"))
pop_recs_split = test_ground_truth_idx.select("session_idx").distinct() \
    .crossJoin(item_popularity_split) \
    .transform(lambda df: exclude_seen_items(df, test_seen_idx, score_col="score", k=TOP_K))

pop_split_metrics = evaluate_recommendations(pop_recs_split, test_ground_truth_idx, k=TOP_K)
pop_split_div = diversity_metrics(pop_recs_split, train_idx)

print("ALS (train/test split, fold-in):")
for metric, value in als_split_metrics.items():
    print(f"  {metric}: {value:.4f}")
for metric, value in als_split_div.items():
    print(f"  {metric}: {value:.4f}")

print("\nPopularity Baseline (on same split):")
for metric, value in pop_split_metrics.items():
    print(f"  {metric}: {value:.4f}")
for metric, value in pop_split_div.items():
    print(f"  {metric}: {value:.4f}")


Train sessions: 17031
Test sessions: 6995
Evaluable test sessions: 5585


26/05/01 12:40:48 WARN TaskSetManager: Stage 12938 contains a task of very large size (1346 KiB). The maximum recommended task size is 1000 KiB.
26/05/01 12:40:49 WARN TaskSetManager: Stage 12961 contains a task of very large size (1346 KiB). The maximum recommended task size is 1000 KiB.
26/05/01 12:40:50 WARN TaskSetManager: Stage 12970 contains a task of very large size (1346 KiB). The maximum recommended task size is 1000 KiB.
26/05/01 12:40:53 WARN TaskSetManager: Stage 13008 contains a task of very large size (1346 KiB). The maximum recommended task size is 1000 KiB.
26/05/01 12:40:54 WARN TaskSetManager: Stage 13037 contains a task of very large size (1346 KiB). The maximum recommended task size is 1000 KiB.


ALS (train/test split, fold-in):
  MRR@10: 0.0867
  Precision@10: 0.0234
  Recall@10: 0.2337
  Coverage: 0.9724
  UniqueRecommendedItems: 211.0000
  Novelty@10: 7.3699

Popularity Baseline (on same split):
  MRR@10: 0.0415
  Precision@10: 0.0119
  Recall@10: 0.1192
  Coverage: 0.2212
  UniqueRecommendedItems: 48.0000
  Novelty@10: 6.0566


In [80]:
# interaction-based item k-NN: learn item-item similarity from shared session behavior

KNN_NEIGHBORS = 50

item_norms = (
    train_idx.groupBy("item_idx")
    .agg(F.sqrt(F.sum(F.pow("click_count", 2))).alias("norm"))
)

train_norm = (
    train_idx.join(item_norms, on="item_idx", how="inner")
    .withColumn("val_norm", F.col("click_count") / F.col("norm"))
)

item_item_sim = (
    train_norm.alias("a")
    .join(train_norm.alias("b"), on="session_idx")
    .filter(F.col("a.item_idx") != F.col("b.item_idx"))
    .groupBy(
        F.col("a.item_idx").alias("seed_idx"),
        F.col("b.item_idx").alias("target_idx"),
    )
    .agg(F.sum(F.col("a.val_norm") * F.col("b.val_norm")).alias("similarity"))
    .withColumn(
        "neighbor_rank",
        F.row_number().over(
            Window.partitionBy("seed_idx").orderBy(F.desc("similarity"), F.asc("target_idx"))
        ),
    )
    .filter((F.col("neighbor_rank") <= KNN_NEIGHBORS) & (F.col("similarity") > 0))
    .drop("neighbor_rank")
)


In [81]:
# Detailed comparison of ALS vs Popularity on the temporal test split
item_popularity = train_idx.groupBy("item_idx").agg(F.sum("click_count").alias("score"))

pop_recs = (
    test_ground_truth_idx.select("session_idx").distinct()
    .crossJoin(item_popularity)
    .transform(lambda df: exclude_seen_items(df, test_seen_idx, score_col="score", k=TOP_K))
)

als_recs = exclude_seen_items(als_recs_test, test_seen_idx, score_col="score", k=TOP_K)

pop_metrics = evaluate_recommendations(pop_recs, test_ground_truth_idx, k=TOP_K)
als_metrics = evaluate_recommendations(als_recs, test_ground_truth_idx, k=TOP_K)

als_div = diversity_metrics(als_recs, train_idx, n_catalog_items=217)
pop_div = diversity_metrics(pop_recs, train_idx, n_catalog_items=217)

print(f"Popularity Coverage: {pop_div['UniqueRecommendedItems']}/217 = {pop_div['Coverage']:.2%}")
print(f"ALS Coverage:        {als_div['UniqueRecommendedItems']}/217 = {als_div['Coverage']:.2%}")

print(f"MRR@10       baseline={pop_metrics['MRR@10']:.4f}  ALS={als_metrics['MRR@10']:.4f}  delta={als_metrics['MRR@10'] - pop_metrics['MRR@10']:+.4f}")
print(f"Precision@10 baseline={pop_metrics['Precision@10']:.4f}  ALS={als_metrics['Precision@10']:.4f}  delta={als_metrics['Precision@10'] - pop_metrics['Precision@10']:+.4f}")
print(f"Recall@10    baseline={pop_metrics['Recall@10']:.4f}  ALS={als_metrics['Recall@10']:.4f}  delta={als_metrics['Recall@10'] - pop_metrics['Recall@10']:+.4f}")
print(f"Coverage     baseline={pop_div['Coverage']:.2%}  ALS={als_div['Coverage']:.2%}  delta={als_div['Coverage'] - pop_div['Coverage']:+.2%}")
print(f"Novelty@10   baseline={pop_div['Novelty@10']:.4f} bits  ALS={als_div['Novelty@10']:.4f} bits  delta={als_div['Novelty@10'] - pop_div['Novelty@10']:+.4f}")


26/05/01 12:41:12 WARN TaskSetManager: Stage 13340 contains a task of very large size (1346 KiB). The maximum recommended task size is 1000 KiB.
26/05/01 12:41:13 WARN TaskSetManager: Stage 13363 contains a task of very large size (1346 KiB). The maximum recommended task size is 1000 KiB.
26/05/01 12:41:14 WARN TaskSetManager: Stage 13372 contains a task of very large size (1346 KiB). The maximum recommended task size is 1000 KiB.
26/05/01 12:41:16 WARN TaskSetManager: Stage 13410 contains a task of very large size (1346 KiB). The maximum recommended task size is 1000 KiB.
26/05/01 12:41:17 WARN TaskSetManager: Stage 13439 contains a task of very large size (1346 KiB). The maximum recommended task size is 1000 KiB.


Popularity Coverage: 48/217 = 22.12%
ALS Coverage:        211/217 = 97.24%
MRR@10       baseline=0.0415  ALS=0.0867  delta=+0.0452
Precision@10 baseline=0.0119  ALS=0.0234  delta=+0.0114
Recall@10    baseline=0.1192  ALS=0.2337  delta=+0.1144
Coverage     baseline=22.12%  ALS=97.24%  delta=+75.12%
Novelty@10   baseline=6.0566 bits  ALS=7.3699 bits  delta=+1.3133


In [82]:
# interaction-based item k-NN for warm sessions using temporal split
from pyspark.sql.functions import col

# use test context as history for recommendations
test_history = (
    item_indexer.transform(test_context_interactions)
    .withColumn("item_idx", F.col("item_idx").cast("int"))
    .withColumn("session_idx", F.col("session_id").cast("int"))
)
seen_items = test_seen_idx

knn_candidate_scores = (
    test_history.alias("h")
    .join(item_item_sim.alias("s"), col("h.item_idx") == col("s.seed_idx"))
    .groupBy(F.col("h.session_idx").alias("session_idx"), F.col("s.target_idx").alias("item_idx"))
    .agg(F.sum(col("h.click_count") * col("s.similarity")).alias("score"))
)

knn_recs = exclude_seen_items(knn_candidate_scores, seen_items, score_col="score", k=TOP_K)

knn_metrics = evaluate_recommendations(knn_recs, test_ground_truth_idx, k=TOP_K)
print("Item-based k-NN (Temporal Split)")
for k_m, v_m in knn_metrics.items():
    print(f"  {k_m}: {v_m:.4f}")

knn_div = diversity_metrics(knn_recs, train_idx, n_catalog_items=217)
print(f"Item k-NN Coverage: {knn_div['UniqueRecommendedItems']}/217 = {knn_div['Coverage']:.2%}")


Item-based k-NN (Temporal Split)
  MRR@10: 0.0966
  Precision@10: 0.0246
  Recall@10: 0.2460


Item k-NN Coverage: 213/217 = 98.16%


In [83]:
# cold sessions have only 1 observed item in training

cold_interactions = (
    cold_train_raw.groupBy("session_id", "item_id")
    .agg(F.count("*").alias("click_count"))
)

cold_train_idx = (
    item_indexer.transform(cold_interactions)
    .withColumn("item_idx", F.col("item_idx").cast("int"))
    .withColumn("session_idx", F.col("session_id").cast("int"))
)

cold_ground_truth_idx = (
    item_indexer.transform(cold_ground_truth)
    .withColumn("item_idx", F.col("item_idx").cast("int"))
    .withColumn("session_idx", F.col("session_id").cast("int"))
)

cold_seen_items = cold_train_idx.select("session_idx", F.col("item_idx").alias("seen_item_idx")).distinct()

pop_recs_cold = (
    cold_ground_truth_idx.select("session_idx").distinct()
    .crossJoin(item_popularity)
    .transform(lambda df: exclude_seen_items(df, cold_seen_items, score_col="score", k=TOP_K))
)

pop_metrics_cold = evaluate_recommendations(pop_recs_cold, cold_ground_truth_idx, k=TOP_K)
pop_div_cold = diversity_metrics(pop_recs_cold, train_idx, n_catalog_items=217)

seq_candidate_scores = (
    cold_train_idx.alias("h")
    .join(item_item_sim.alias("s"), col("h.item_idx") == col("s.seed_idx"))
    .groupBy(F.col("h.session_idx").alias("session_idx"), F.col("s.target_idx").alias("item_idx"))
    .agg(F.sum(col("h.click_count") * col("s.similarity")).alias("score"))
)

seq_recs = exclude_seen_items(seq_candidate_scores, cold_seen_items, score_col="score", k=TOP_K)

seq_metrics = evaluate_recommendations(seq_recs, cold_ground_truth_idx, k=TOP_K)
seq_div = diversity_metrics(seq_recs, train_idx, n_catalog_items=217)

print(f"Cold-start Evaluation (N={cold_ground_truth_idx.select('session_idx').distinct().count()} sessions):")
print(f"  Popularity MRR@10: {pop_metrics_cold['MRR@10']:.4f}")
print(f"  Item Seq-Warmup MRR@10: {seq_metrics['MRR@10']:.4f}")
print(f"  Popularity Recall@10: {pop_metrics_cold['Recall@10']:.4f}")
print(f"  Item Seq-Warmup Recall@10: {seq_metrics['Recall@10']:.4f}")
print(f"  Popularity Novelty@10: {pop_div_cold['Novelty@10']:.4f} bits")
print(f"  Item Seq-Warmup Novelty@10: {seq_div['Novelty@10']:.4f} bits")


Cold-start Evaluation (N=3320 sessions):
  Popularity MRR@10: 0.0545
  Item Seq-Warmup MRR@10: 0.1450
  Popularity Recall@10: 0.1639
  Item Seq-Warmup Recall@10: 0.3434
  Popularity Novelty@10: 6.0233 bits
  Item Seq-Warmup Novelty@10: 7.2209 bits


In [84]:



# Word2Vec / Item2Vec with Exponential Weighted Pooling
from pyspark.ml.feature import Word2Vec
from pyspark.ml.functions import vector_to_array
from pyspark.sql import functions as F
from pyspark.sql.window import Window

ITEM2VEC_VECTOR_SIZE = 32
ITEM2VEC_WINDOW_SIZE = 3
ITEM2VEC_MAX_ITER = 20
ITEM2VEC_MIN_COUNT = 5
ITEM2VEC_DECAY = 0.8




def build_ordered_item_sequences(history_raw_df):
    return (
        history_raw_df.groupBy("session_id")
        .agg(
            F.sort_array(
                F.collect_list(F.struct(F.col("order"), F.col("item_id").cast("string").alias("item_id_str")))
            ).alias("ordered_items")
        )
        .withColumn("item_sequence", F.expr("transform(ordered_items, x -> x.item_id_str)"))
        .select("session_id", "item_sequence")
    )

train_sequences = build_ordered_item_sequences(train_raw_split)

item2vec = Word2Vec(
    vectorSize=ITEM2VEC_VECTOR_SIZE,
    minCount=ITEM2VEC_MIN_COUNT,
    maxIter=ITEM2VEC_MAX_ITER,
    windowSize=ITEM2VEC_WINDOW_SIZE,
    inputCol="item_sequence",
    outputCol="session_vector",
    seed=42,
)
item2vec_model = item2vec.fit(train_sequences)

item_label_df = spark.createDataFrame(
    [(int(idx), label) for idx, label in enumerate(item_indexer.labels)],
    ["item_idx", "word"],
)

item2vec_item_vectors = (
    item2vec_model.getVectors()
    .join(item_label_df, on="word", how="inner")
    .select("item_idx", F.col("vector").alias("item_vector"))
    .withColumn("item_array", vector_to_array("item_vector"))
    .withColumn("item_norm", F.expr("sqrt(aggregate(item_array, 0D, (acc, x) -> acc + x * x))"))
    .select("item_idx", "item_array", "item_norm")
    .cache()
)

def build_item2vec_recommendations(history_raw_df, seen_items_df, candidate_sessions_df, decay=ITEM2VEC_DECAY, k=TOP_K):
    # exponential weighted pooling: later items have more weight
    w_spec = Window.partitionBy("session_id").orderBy("order")
    w_session = Window.partitionBy("session_id")
    
    session_items_weighted = (
        history_raw_df
        .withColumn("pos", F.row_number().over(w_spec))
        .withColumn("max_pos", F.max("pos").over(w_session))
        .withColumn("weight", F.pow(F.lit(decay), F.col("max_pos") - F.col("pos")))
    )
    
    session_vectors = (
        item_indexer.transform(session_items_weighted)
        .withColumn("item_idx", F.col("item_idx").cast("int"))
        .join(item2vec_item_vectors, on="item_idx", how="inner")
        .withColumn("weighted_array", F.expr("transform(item_array, x -> x * weight)"))
        .groupBy("session_id")
        .agg(F.expr(f"aggregate(collect_list(weighted_array), cast(array_repeat(0D, {ITEM2VEC_VECTOR_SIZE}) as array<double>), (acc, x) -> zip_with(acc, x, (a, b) -> a + b))").alias("session_array"))
        .withColumn("session_idx", F.col("session_id").cast("int"))
        .withColumn("session_norm", F.expr("sqrt(aggregate(session_array, 0D, (acc, x) -> acc + x * x))"))
        .join(candidate_sessions_df.select("session_idx").distinct(), on="session_idx", how="inner")
        .select("session_idx", "session_array", "session_norm")
    )

    candidate_scores = (
        session_vectors.crossJoin(item2vec_item_vectors)
        .withColumn(
            "dot",
            F.expr("aggregate(zip_with(session_array, item_array, (x, y) -> x * y), 0D, (acc, x) -> acc + x)")
        )
        .withColumn(
            "score",
            F.when(
                (F.col("session_norm") > 0.0) & (F.col("item_norm") > 0.0),
                F.col("dot") / (F.col("session_norm") * F.col("item_norm")),
            ).otherwise(F.lit(0.0)),
        )
        .select("session_idx", "item_idx", "score")
    )
    return exclude_seen_items(candidate_scores, seen_items_df, score_col="score", k=k)

item2vec_recs = (
    build_item2vec_recommendations(
        test_context_raw,
        test_seen_idx,
        test_ground_truth_idx.select("session_idx").distinct(),
        decay=ITEM2VEC_DECAY,
        k=TOP_K,
    )
)
item2vec_metrics = evaluate_recommendations(item2vec_recs, test_ground_truth_idx, k=TOP_K)
item2vec_div = diversity_metrics(item2vec_recs, train_idx, n_catalog_items=217)

print("Item2Vec warm-session recommender (Weighted Pooling)")
for metric_name, metric_value in item2vec_metrics.items():
    print(f"  {metric_name}: {metric_value:.4f}")
print(f"  Coverage: {item2vec_div['UniqueRecommendedItems']}/217 = {item2vec_div['Coverage']:.2%}")
print(f"  Novelty@10: {item2vec_div['Novelty@10']:.4f} bits")


Item2Vec warm-session recommender (Weighted Pooling)
  MRR@10: 0.1031
  Precision@10: 0.0286
  Recall@10: 0.2858
  Coverage: 213/217 = 98.16%
  Novelty@10: 7.6794 bits


In [85]:
# Final evaluation results
evaluation_metrics = {
    "warm_sessions": {
        "popularity": {
            "MRR@10": round(pop_split_metrics["MRR@10"], 4),
            "Recall@10": round(pop_split_metrics["Recall@10"], 4),
            "Coverage": round(pop_split_div["Coverage"], 4),
            "Novelty@10": round(pop_split_div["Novelty@10"], 4),
        },
        "als": {
            "MRR@10": round(als_split_metrics["MRR@10"], 4),
            "Recall@10": round(als_split_metrics["Recall@10"], 4),
            "Coverage": round(als_split_div["Coverage"], 4),
            "Novelty@10": round(als_split_div["Novelty@10"], 4),
        },
        "knn": {
            "MRR@10": round(knn_metrics["MRR@10"], 4),
            "Recall@10": round(knn_metrics["Recall@10"], 4),
            "Coverage": round(knn_div["Coverage"], 4),
            "Novelty@10": round(knn_div["Novelty@10"], 4),
        },
        "item2vec": {
            "MRR@10": round(item2vec_metrics["MRR@10"], 4),
            "Recall@10": round(item2vec_metrics["Recall@10"], 4),
            "Coverage": round(item2vec_div["Coverage"], 4),
            "Novelty@10": round(item2vec_div["Novelty@10"], 4),
        }
    },
    "cold_sessions": {
        "popularity": {
            "MRR@10": round(pop_metrics_cold["MRR@10"], 4),
            "Recall@10": round(pop_metrics_cold["Recall@10"], 4),
            "Coverage": round(pop_div_cold["Coverage"], 4),
            "Novelty@10": round(pop_div_cold["Novelty@10"], 4),
        },
        "sequential_warmup": {
            "MRR@10": round(seq_metrics["MRR@10"], 4),
            "Recall@10": round(seq_metrics["Recall@10"], 4),
            "Coverage": round(seq_div["Coverage"], 4),
            "Novelty@10": round(seq_div["Novelty@10"], 4),
        },
        "item2vec": {
            "MRR@10": round(item2vec_metrics_cold["MRR@10"], 4),
            "Recall@10": round(item2vec_metrics_cold["Recall@10"], 4),
            "Coverage": round(item2vec_div_cold["Coverage"], 4),
            "Novelty@10": round(item2vec_div_cold["Novelty@10"], 4),
        },
    }
}

print("\n---Final evaluation metrics---")
print(evaluation_metrics)

import json
import os

top_items_list = (
    item_popularity.orderBy(F.desc("score"), F.asc("item_idx"))
    .limit(10)
    .select("item_idx", "score")
    .collect()
)
labels = item_indexer.labels
top_items_data = [
    {"item_id": labels[int(row.item_idx)], "views": int(row.score)}
    for row in top_items_list
]

als_params = {
    "rank": ALS_RANK,
    "alpha": ALS_ALPHA,
    "regParam": ALS_REG_PARAM
}

item2vec_params = {
    "vectorSize": ITEM2VEC_VECTOR_SIZE,
    "windowSize": ITEM2VEC_WINDOW_SIZE,
    "maxIter": ITEM2VEC_MAX_ITER,
    "minCount": ITEM2VEC_MIN_COUNT,
}

output_path = "../evaluation_metrics/recommender_metrics.json"
os.makedirs(os.path.dirname(output_path), exist_ok=True)

full_metrics = {
    "metrics": evaluation_metrics,
    "top_items": top_items_data,
    "params": {
        "als": als_params,
        "item2vec": item2vec_params,
    },
}

with open(output_path, "w") as f:
    json.dump(full_metrics, f, indent=4)

print(f"Metrics saved to {output_path}")



---Final evaluation metrics---
{'warm_sessions': {'popularity': {'MRR@10': 0.0415, 'Recall@10': 0.1192, 'Coverage': 0.2212, 'Novelty@10': 6.0566}, 'als': {'MRR@10': 0.0867, 'Recall@10': 0.2337, 'Coverage': 0.9724, 'Novelty@10': 7.3699}, 'knn': {'MRR@10': 0.0966, 'Recall@10': 0.246, 'Coverage': 0.9816, 'Novelty@10': 7.1585}, 'item2vec': {'MRR@10': 0.1031, 'Recall@10': 0.2858, 'Coverage': 0.9816, 'Novelty@10': 7.6794}}, 'cold_sessions': {'popularity': {'MRR@10': 0.0545, 'Recall@10': 0.1639, 'Coverage': 0.0507, 'Novelty@10': 6.0233}, 'sequential_warmup': {'MRR@10': 0.145, 'Recall@10': 0.3434, 'Coverage': 0.9816, 'Novelty@10': 7.2209}}}
Metrics saved to ../evaluation_metrics/recommender_metrics.json
